<a href="https://colab.research.google.com/github/tkoziara/Piper-Tools/blob/main/Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 0) Verify Colab GPU and environment

In [ ]:
import torch, sys
print('Python', sys.version.split()[0])
print('PyTorch', torch.__version__)
print('CUDA available', torch.cuda.is_available())
# show GPU info if available
if torch.cuda.is_available():
    try:
        import subprocess
        print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader']).decode().strip())
    except Exception as e:
        print('nvidia-smi failed:', e)

## 1) Bootstrap environment (run once)

In [ ]:
%%bash
set -e
echo 'Colab bootstrap: installing build deps and piper1-gpl (editable)'
python3 -m pip install --upgrade pip setuptools wheel
python3 -m pip install --upgrade scikit-build cmake ninja wheel setuptools
if [ ! -d /content/piper1-gpl ]; then
  if git ls-remote git@github.com:tkoziara/piper1-gpl.git >/dev/null 2>&1; then
    git clone git@github.com:tkoziara/piper1-gpl.git /content/piper1-gpl
  else
    git clone https://github.com/tkoziara/piper1-gpl.git /content/piper1-gpl
  fi
fi
cd /content/piper1-gpl
python3 -m pip install -e ".[train]" || true
bash ./build_monotonic_align.sh || true
python3 setup.py build_ext --inplace -v || true
python3 -m pip install --upgrade onnxscript flask openai-whisper soundfile librosa lightning pytorch-lightning pysilero-vad pathvalidate jsonargparse[signatures] || true
python3 - <<'PY'
import torch, sys
print('Python', sys.version.split()[0])
print('PyTorch', torch.__version__)
print('CUDA available', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA device', torch.cuda.get_device_name(0))
PY

## 2) Mount Drive and set paths/hyperparameters

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
VOICE_NAME = 'tomek_pl'
# Maximum number of training epochs (set to None to leave Trainer default)
MAX_EPOCHS = 100
ESPEAK_VOICE = 'pl'
SAMPLE_RATE = 22050
# Primary batch size used by the notebook; training launcher will use this as an override if set
BATCH_SIZE = 16
# Optional overrides for Colab/T4 memory tuning (can be adjusted here)
NUM_WORKERS = 2
SEGMENT_SIZE = 6144
PRECISION = '16'
# Checkpoint monitor settings: enable/disable and poll interval (seconds)
ENABLE_CHECKPOINT_MONITOR = True
CHECKPOINT_POLL_INTERVAL = 10
DRIVE_ROOT = Path('/content/drive/MyDrive/tts_data')
DATA_ROOT = DRIVE_ROOT / VOICE_NAME
AUDIO_DIR = DATA_ROOT / 'wavs'
CSV_PATH = DATA_ROOT / 'metadata.csv'
CACHE_DIR = Path('/content/piper_cache')
CKPT_PATH = DRIVE_ROOT / f"{VOICE_NAME}-latest.ckpt"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print('CSV exists', CSV_PATH.exists())
print('Audio dir exists', AUDIO_DIR.exists())
print('Checkpoint exists', CKPT_PATH.exists())

## 3) Sanitize checkpoint and run training

In [ ]:
# Cell 3 replacement — VAD + checkpoint migration + memory-reduction training launcher
import sys, os, torch, pathlib, tempfile, subprocess, textwrap
from pathlib import Path

# environment tweaks to reduce CUDA fragmentation / OOMs
os.environ.setdefault("PYTORCH_ALLOC_CONF", "max_split_size_mb:128,allocator:default")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128")
torch.backends.cudnn.benchmark = False

# ensure piper source is importable
PIPER_SRC = Path('/content/piper1-gpl/src')
if PIPER_SRC.exists():
    sys.path.insert(0, str(PIPER_SRC))

torch.serialization.add_safe_globals([pathlib.PosixPath])

print("DEBUG: VOICE_NAME =", globals().get('VOICE_NAME'))
print("DEBUG: DATA_ROOT  =", globals().get('DATA_ROOT'))
print("DEBUG: CKPT_PATH  =", globals().get('CKPT_PATH'))

DATA_ROOT = Path(DATA_ROOT)
CONFIG_PATH = DATA_ROOT / f"{VOICE_NAME}.json"
if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Missing required config: {CONFIG_PATH}")

CKPT_PATH = Path(CKPT_PATH) if 'CKPT_PATH' in globals() and CKPT_PATH is not None else None

# helper: detect PL version for checkpoint injection
def _get_pl_version():
    try:
        import lightning.pytorch as _lp
        return getattr(_lp, "__version__", None) or getattr(_lp, "version", None)
    except Exception:
        try:
            import lightning as _l
            return getattr(_l, "__version__", None) or getattr(_l, "version", None)
        except Exception:
            return "unknown"

pl_version = _get_pl_version()

# Patch dataset.py for VAD compatibility if needed
dataset_file = PIPER_SRC.parent / "src" / "piper" / "train" / "vits" / "dataset.py"
if not dataset_file.exists():
    dataset_file = PIPER_SRC / "piper" / "train" / "vits" / "dataset.py"

if dataset_file.exists():
    text = dataset_file.read_text(encoding="utf-8")
    marker = "# BEGIN ADDED: pysilero_vad compatibility shim"
    if marker not in text:
        import_line = "from pysilero_vad import SileroVoiceActivityDetector"
        if import_line in text:
            shim = f"""
{marker}
# Compatibility shim: ensure SileroVoiceActivityDetector has a `process_array` method
try:
    if not hasattr(SileroVoiceActivityDetector, "process_array"):
        _orig_vad_cls = SileroVoiceActivityDetector
        class SileroVoiceActivityDetector(_orig_vad_cls):
            def process_array(self, chunk):
                try:
                    if hasattr(self, "is_speech"):
                        res = self.is_speech(chunk)
                        return float(res) if not isinstance(res, (list, tuple)) else float(1.0 if len(res) else 0.0)
                    if hasattr(self, "get_speech_timestamps"):
                        res = self.get_speech_timestamps(chunk)
                        return float(1.0 if res else 0.0)
                    if hasattr(self, "get_speech_probabilities"):
                        res = self.get_speech_probabilities(chunk)
                        if isinstance(res, (list, tuple)):
                            return float(res[0]) if res else 0.0
                        return float(res)
                except Exception:
                    pass
                import numpy as _np
                arr = _np.asarray(chunk)
                if arr.size == 0:
                    return 0.0
                energy = float(_np.mean(_np.abs(arr)))
                return min(1.0, energy / 0.01)
except Exception:
    pass
# END ADDED: pysilero_vad compatibility shim
"""
            new_text = text.replace(import_line, import_line + "\n" + shim)
            dataset_file.write_text(new_text, encoding="utf-8")
            print("DEBUG: patched dataset.py for VAD compatibility at", dataset_file)
    else:
        print("DEBUG: dataset.py already patched for VAD compatibility")
else:
    print("WARNING: could not find dataset.py at expected locations; skipping patch")

# Save weights-only checkpoint and inject PL version to avoid migration errors
use_ckpt = None
tmp_ckpt = None
if CKPT_PATH and CKPT_PATH.exists():
    print("DEBUG: loading checkpoint for weights extraction:", CKPT_PATH)
    ckpt = torch.load(str(CKPT_PATH), map_location='cpu', weights_only=False)
    ckpt.pop('hyper_parameters', None)
    ckpt.pop('hparams', None)
    minimal = {k: v for k, v in ckpt.items() if k in ('state_dict', 'optimizer_states', 'lr_schedulers', 'epoch', 'global_step')}
    if not minimal:
        minimal = ckpt
    minimal.setdefault('pytorch-lightning_version', pl_version)
    minimal.setdefault('pytorch_lightning_version', pl_version)
    fd, tmp_path = tempfile.mkstemp(suffix='.ckpt', prefix='weights_only_')
    os.close(fd)
    tmp_ckpt = Path(tmp_path)
    torch.save(minimal, str(tmp_ckpt))
    use_ckpt = tmp_ckpt
    print("DEBUG: saved weights-only temporary checkpoint:", tmp_ckpt)
else:
    print("INFO: no checkpoint present (will start fresh)")

# Memory-reduction overrides (tweak these if you still hit OOM)
# These honor the top-level notebook variables when present
batch_override = int(globals().get('BATCH_SIZE', 4))
num_workers_override = int(globals().get('NUM_WORKERS', 0))
segment_size_override = int(globals().get('SEGMENT_SIZE', 4096))
precision = str(globals().get('PRECISION', '16'))

# Build CLI args (use overrides)
cli_args = [
    'fit',
    '--data.voice_name', VOICE_NAME,
    '--data.csv_path', str(DATA_ROOT / 'metadata.csv'),
    '--data.audio_dir', str(DATA_ROOT / 'wavs'),
    '--data.cache_dir', str(globals().get('CACHE_DIR', '/content/piper_cache')),
    '--data.espeak_voice', globals().get('ESPEAK_VOICE', 'pl'),
    '--data.batch_size', str(batch_override),
    '--data.num_workers', str(num_workers_override),
    '--data.config_path', str(CONFIG_PATH),
    '--model.sample_rate', str(globals().get('SAMPLE_RATE', 22050)),
    '--model.segment_size', str(segment_size_override),
    '--trainer.precision', precision,
]

if use_ckpt:
    cli_args += ['--ckpt_path', str(use_ckpt), '--weights_only', 'true']

if 'MAX_EPOCHS' in globals() and MAX_EPOCHS is not None:
    cli_args += ['--trainer.max_epochs', str(MAX_EPOCHS)]

if torch.cuda.is_available():
    cli_args += ['--trainer.accelerator', 'gpu', '--trainer.devices', '1']

print("DEBUG: final CLI args:", cli_args)

# Write launcher and run subprocess (child inherits env tweaks above)
launcher = textwrap.dedent(f"""
    import runpy, sys, pathlib, torch
    torch.serialization.add_safe_globals([pathlib.PosixPath])
    sys.argv = ['piper.train'] + {repr(cli_args)}
    runpy.run_module('piper.train', run_name='__main__')
""")
with tempfile.NamedTemporaryFile('w', suffix='.py', delete=False) as f:
    f.write(launcher)
    launcher_path = f.name

print("DEBUG: launcher written to", launcher_path)
python_exe = sys.executable
cmd = [python_exe, '-u', launcher_path]
print("DEBUG: launching training subprocess:", ' '.join(cmd))

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                        cwd=str(PIPER_SRC.parent if PIPER_SRC.exists() else os.getcwd()), env=os.environ.copy())
try:
    for line in proc.stdout:
        print(line, end='')
    ret = proc.wait()
finally:
    try:
        proc.stdout.close()
    except Exception:
        pass

print("DEBUG: subprocess exited with", ret)

# cleanup
if tmp_ckpt and tmp_ckpt.exists():
    try:
        tmp_ckpt.unlink()
        print("DEBUG: removed temporary ckpt", tmp_ckpt)
    except Exception as e:
        print("WARNING: could not remove temp ckpt:", e)

try:
    if os.path.exists(launcher_path):
        os.remove(launcher_path)
        print("DEBUG: removed launcher", launcher_path)
except Exception:
    pass

if ret != 0:
    raise subprocess.CalledProcessError(ret, cmd)

## 4) Export lastet checkpoint to Google Drive

Save the latest checkpoint into your Google Drive for training restart and local export to onnx.
Once the local environment is set up via:

```bash
./bootstrap.sh
source .venv/bin/activate
```

You can export the latest checkpoint from Google Drive to your local environment via:

```bash
python train.py export --checkpoint <checkpoint_in_google_drive> --output <model_name>.onnx
python synth.py --model <model_name>.onnx --play --text "Your text here"
```

In [ ]:
# Copy latest checkpoint to Drive for safekeeping (run in Colab)
from pathlib import Path
import shutil, sys
from datetime import datetime

try:
    DATA_ROOT, DRIVE_ROOT, VOICE_NAME
except NameError:
    raise RuntimeError("Run cell 2 first to set DATA_ROOT, DRIVE_ROOT and VOICE_NAME.")

search_roots = [
    DATA_ROOT / "tts_output",
    DATA_ROOT,
    Path("/content/piper1-gpl/lightning_logs"),
    Path("/content/piper1-gpl"),
    Path("/tmp"),
    Path("/content"),
]

cands = []
for root in search_roots:
    if not root.exists():
        continue
    for p in root.rglob("*.ckpt"):
        cands.append(p.resolve())

if not cands:
    raise FileNotFoundError("No .ckpt files found under expected locations.")

latest = max(cands, key=lambda p: p.stat().st_mtime)
print("Latest checkpoint found:", latest)

# ensure Drive dir exists
dst_dir = Path(DRIVE_ROOT)
dst_dir.mkdir(parents=True, exist_ok=True)

# Human-readable timestamp and single destination filename
timestamp = datetime.utcnow().strftime("%Y-%m-%d_%H-%M-%S")
dst_name = f"{VOICE_NAME}-{timestamp}.ckpt"
dst_path = dst_dir / dst_name
print("Copying to Drive:", dst_path)
shutil.copy2(str(latest), str(dst_path))

# Also write a stable 'latest' copy for quick reference
stable = dst_dir / f"{VOICE_NAME}-latest.ckpt"
try:
    if stable.exists():
        stable.unlink()
    shutil.copy2(str(dst_path), str(stable))
    print("Also wrote stable copy:", stable)
except Exception:
    print("Could not write stable copy (ignored).")

print("Done. Drive copy:", dst_path, f"({dst_path.stat().st_size/1024/1024:.1f} MB)")